# D01-数据接入与字段画像

手上是 `data/hotel.db`：119,390 条酒店预订记录，2015-07-01 至 2017-08-31。
这一本不做结论，只把数据看清楚：**有哪些字段、缺在哪、时间覆盖多长、每月的量和取消率长什么样**。
看清楚了，D02 才知道该查什么。

## 手上有什么

单文件 SQLite 库，三样东西：

| 对象 | 是什么 | 本项目的用法 |
| --- | --- | --- |
| `bookings` | 原始层，119,390 行 × 37 列 | 只读，一个值都不改 |
| `column_dictionary` | 37 行字段字典 | 取中文名，给图上的标签用 |
| `v_monthly_kpi` 等 3 个视图 | 按月/按渠道的查询定义 | 口径参考，不落库 |

37 列 = 1 个代理主键（`booking_id`）+ 32 个业务字段 + 4 个派生列
（`arrival_month_no`、`arrival_date`、`total_nights`、`total_guests`）。
派生列由入库脚本拼好，**原字段原样保留**——这是 DQ-05 的处理结果，后面会讲为什么不能只留派生列。

### 环境准备

In [ ]:
# ── 第一格：环境准备（每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hotel_insight import config, db, quality, viz
from hotel_insight.viz import MAIN, MAIN_LIGHT, MAIN_MID, MAIN_DARK, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT   # 配色常量

viz.apply_theme()                              # 字体、配色、坐标轴样式：一行
con = db.connect()                             # 路径从包里来：notebook 在哪打开都找得到库
NB = "D01_数据接入"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"     # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
# 自检：版本、中文字体、库大小——环境不对，后面的图会全变方框
import sys, matplotlib
from matplotlib import font_manager

installed = {f.name for f in font_manager.fontManager.ttflist}
cn_font = [f for f in plt.rcParams["font.family"] if f in installed and f != "DejaVu Sans"]

print(f"Python      {sys.version.split()[0]}")
print(f"pandas      {pd.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"SQLite      {sqlite3.sqlite_version}")
print(f"中文字体    {cn_font}")
print(f"数据库      {config.DB_PATH.name}（{config.DB_PATH.stat().st_size / 1e6:.0f} MB）")

## 数据接入与首次对账

In [ ]:
raw = pd.read_sql(f"SELECT * FROM {config.TABLE}", con)
print(f"{len(raw):,} 行 × {raw.shape[1]} 列")

# 对账：行数、列数、列序三项都必须是契约里的值，否则下游全部读错位
assert raw.shape == (119_390, 37), f"行列数不对：{raw.shape}，应该是 (119390, 37)"
assert list(raw.columns) == ["booking_id", *config.BUSINESS_COLS, *config.DERIVED_COLS], "列序与契约不符"
print("对账通过：119,390 行 × 37 列，列序与契约一致")

行数与列序对上，才谈得上后面的分析。这三条断言是**先手**：数据库换了一批数据，
它们会立刻失败，而不是等到图画出来才发现数字怪。

## 字段画像：每个字段长什么样

In [ ]:
prof = quality.profile(raw)
p = prof.set_index("字段")

# 对账：只有 4 个字段有缺失，其中 company_id 是唯一整列不可用的
assert (prof["缺失数"] > 0).sum() == 4, "有缺失的字段数变了"
assert p.loc["company_id", "缺失数"] == 112_593
assert p.loc["agent_id", "缺失数"] == 16_340
assert p.loc["country", "缺失数"] == 488
assert p.loc["children", "缺失数"] == 4
print(f"对账通过：{int((prof['缺失数'] == 0).sum())} 个字段零缺失，4 个字段有缺失")

prof

缺失呈**两极分布**：33 个字段完全没有缺失，4 个字段有缺失，其中 `company_id` 缺 94%。
不存在"每列都缺一点"的弥散型缺失——这对建模是好消息，但也意味着 `company_id`
只剩"是否协议客户"这一种用法。

### 图 D01_01：缺失率

In [ ]:
# ── 出图 D01_01：缺失率横条 ──────────────────────────────────────────
# 为什么用横条：字段名是中文长标签，竖着排会挤成一团；横条让标签顺着读。
# 为什么只画有缺失的 4 个字段：其余 33 列都是 0，画出来是 33 根空白条，把要看的东西挤没了。
# 规范落在哪几行：横轴从 0 开始；红色只给"整列不可用"的那一个；条尾直接写数，不用图例。
dim = pd.read_sql("SELECT * FROM column_dictionary", con)
cn = dict(zip(dim["column_name"], dim["chinese_name"]))        # 字段名 → 中文名

miss = prof[prof["缺失数"] > 0].sort_values("缺失率")
colors = [RED if r > 0.5 else MAIN for r in miss["缺失率"]]
y = np.arange(len(miss))

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.barh(y, miss["缺失率"], color=colors, height=0.55)
for yi, n, r in zip(y, miss["缺失数"], miss["缺失率"]):
    ax.text(r + 0.012, yi, f"{n:,} 行（{quality.fmt_pct(r)}）", va="center", fontsize=10, color=INK)

ax.set_yticks(y, [f"{cn.get(f, f)}　{f}" for f in miss["字段"]], fontsize=10.5)
ax.set_xlim(0, 1.18)
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlabel("缺失率（分母 = 119,390 行）")
ax.grid(axis="x", visible=True)
ax.grid(axis="y", visible=False)

# ---- 标题、副标题：数字全部由代码算 ----
top = miss.iloc[-1]
rest_rate = miss.iloc[:-1]["缺失数"].sum() / len(raw)
title = f"{top['字段']} 缺 {quality.fmt_pct(top['缺失率'])}，其余三个字段合计 {quality.fmt_pct(rest_rate)}"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = ("口径：缺失 = NULL，或去首尾空白后为空；37 列中 33 列零缺失，有缺失的只有 4 列\n"
            "红色 = 缺失率超过 50%，该字段不能作为分析维度；横轴从 0 开始")
viz.finish(fig, title, subtitle, SOURCE, "D01_01_字段缺失率.png")
plt.show()

In [ ]:
# ---- 读回断言：把数从图对象里读回来核对，靠代码不靠眼睛 ----
widths = [b.get_width() for b in ax.containers[0]]
assert np.allclose(widths, miss["缺失率"].to_numpy()), "条形长度和画像表对不上"
assert quality.fmt_pct(top["缺失率"]) in fig.texts[0].get_text(), "标题里的比例不是从数据算的"
print("读回断言通过：条形长度 = 画像表，标题里的比例来自数据")

## 时间覆盖：26 个月，首尾年份不完整

先按月聚合，看量级和取消率。注意这里用的是**含重复口径**：31,994 行完全重复还没去掉，
所以柱高代表"记录条数"，不等于"订单数"。这条口径差异是 D02 的主线。

In [ ]:
monthly = (raw.assign(月份=raw["arrival_date"].str[:7])
              .groupby("月份")
              .agg(记录数=("booking_id", "size"), 取消率=("is_canceled", "mean")))

assert len(monthly) == 26, f"月份数不对：{len(monthly)}"
peak, low = monthly["记录数"].idxmax(), monthly["记录数"].idxmin()
peak_n, low_n = int(monthly.loc[peak, "记录数"]), int(monthly.loc[low, "记录数"])
ratio = peak_n / low_n
print(f"{len(monthly)} 个月；{peak} 最多 {peak_n:,} 条，{low} 最少 {low_n:,} 条")

monthly.head()

### 图 D01_02：每月记录数与取消率

In [ ]:
# ── 出图 D01_02：月度记录数与取消率 ────────────────────────────────────
# 为什么上下两个面板而不是双 Y 轴：双轴的两条线交叉在哪、谁高谁低，全由刻度决定，容易误导（规范禁止）。
# 为什么用柱 + 折线：记录数是"量"，从 0 起；取消率是"比例"，看走势。
# 规范落在哪几行：共用横轴、各自独立纵轴；横轴每 3 个月一个刻度；峰值用主色、其余灰色。
x = np.arange(len(monthly))
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.6), sharex=True,
                               height_ratios=[1.5, 1], layout="constrained")

colors = [MAIN if m == peak else GRAY for m in monthly.index]
ax1.bar(x, monthly["记录数"], color=colors, width=0.72)
ax1.set_ylim(0, peak_n * 1.2)
ax1.annotate(f"{peak} 最多 {peak_n:,} 条", (monthly.index.get_loc(peak), peak_n),
             xytext=(0, 6), textcoords="offset points", ha="center", va="bottom",
             fontsize=10, color=MAIN, fontweight="bold")
ax1.text(monthly.index.get_loc(low), low_n * 1.05, f"最低 {low_n:,} 条",
         ha="center", va="bottom", fontsize=9.5, color=MUTED)
ax1.set_ylabel("预订记录数（条）")
ax1.set_title("每月预订记录数", loc="left")

ax2.plot(x, monthly["取消率"], color=MAIN, lw=2, marker="o", ms=3.5)
ax2.set_ylim(0, monthly["取消率"].max() * 1.35)
ax2.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax2.set_ylabel("取消率（%）")
ax2.set_xlabel("到达月份")
ax2.set_title("每月取消率", loc="left")

step = 3
ax2.set_xticks(x[::step], monthly.index[::step], rotation=45, ha="right")

# ---- 标题、副标题 ----
title = f"{peak} 最多 {peak_n:,} 条，是 {low} 的 {ratio:.1f} 倍"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = ("含重复口径：31,994 行完全重复未去重，柱高会把真实订单量放大；跨度 26 个月\n"
            "取消率 = 当月 is_canceled 均值；2015 年缺 1—6 月、2017 年止于 8 月，不能做整年同比")
viz.finish(fig, title, subtitle, SOURCE, "D01_02_月度记录数与取消率.png")
plt.show()

In [ ]:
# ---- 读回断言：柱高、折线点数与中间表逐项对齐 ----
heights = [b.get_height() for b in ax1.containers[0]]
assert np.allclose(heights, monthly["记录数"].to_numpy()), "柱高和月度表对不上"
assert len(ax2.lines[0].get_ydata()) == len(monthly), "折线点数与月份数不一致"
assert f"{peak_n:,} 条" in fig.texts[0].get_text(), "标题里的记录数不是从数据算的"
print("读回断言通过：柱高 = 月度表，折线点数 = 月份数")

## 这一本看到了什么

| 编号 | 发现 | 证据 | 影响 / 决定 |
| --- | --- | --- | --- |
| F1 | 119,390 行 × 37 列，列序与契约一致 | 首次对账断言 | 后续所有分析以此为准 |
| F2 | 33 个字段零缺失，`company_id` 缺 94% | 图 D01_01 | 该字段只能降级为二值标记（DQ-02） |
| F3 | 时间跨度 26 个月，2015 年缺上半年、2017 年止于 8 月 | 月度聚合 | 只有 2016 年是完整年，不能做整年同比（DQ-01、DQ-13） |
| F4 | 月度记录数最高与最低相差约 2.8 倍，取消率另有一条波动 | 图 D01_02 | 量与率不同步，取消行为另有驱动因素 |
| F5 | 含重复口径的取消率 37%，去重口径 27% | D02 详算 | **报数必须注明口径**，否则同一指标能差 10 个百分点 |

下一本 D02 做 16 项质量检查、四步清洗，并把结论登记进 `dq_check_result`。

In [ ]:
con.close()   # 用完关闭数据库连接